# 🛸 M5 — Drone Fleet Monitoring Dashboard (bản chạy local trong VS Code)

Bản này thay cho `Drone_pynb.ipynb` cũ, và đã bỏ hết phần chỉ dùng được trên
Google Colab (`google.colab.files.upload()`, tunnel `bore.pub`...) để chạy
gọn trong **VS Code (Jupyter extension)** trên máy local.

Những gì đã đổi so với bản cũ:

1. **Không tự viết lại `schema.py` / `Dronedemo.py` / `m4_ml_engineer.py` bằng `%%writefile`** — các cell đó là bản DEMO tự chế, lệch với bản đã fix của nhóm. Notebook này dùng trực tiếp 6 file thật đặt **cùng thư mục với notebook**: `schema.py`, `m1_streaming.py`, `stochastic_simulator.py`, `kalman_filter.py`, `m2_analytics_pipeline.py`, `m4_ml_engineer.py`.
2. **`app.py` đổi `"DroneFleetDB.db"` → `"drone_fleet.db"`** — khớp tên DB mà M1/M2/M4 thật sự dùng.
3. **Danh sách drone trong sidebar lấy động** bằng `SELECT drone_id FROM dim_drones` thay vì hardcode `["DRONE_001", "DRONE_002"]` (hệ thống có 3 drone).
4. **Bỏ fallback `generate_mock_telemetry_row()`** khi DB lỗi/rỗng — dashboard giờ hiển thị thông báo "chưa có dữ liệu" thay vì giả vờ có dữ liệu (tránh gây hiểu lầm số liệu thật).
5. **Chạy Streamlit local** thay vì Bore tunnel: mở thẳng `http://localhost:8501` trên máy bạn, không cần expose ra internet.

Những điểm làm đúng của bản cũ **vẫn được giữ nguyên**:
- Đọc Gold đúng cách: `SELECT * FROM fact_gold_summary WHERE drone_id = ? ORDER BY window_end DESC LIMIT 1`.
- Cơ chế polling `time.sleep(3); st.rerun()` cho near real-time.
- Giao diện đa ngôn ngữ VI/EN.

**Nhắc lại vai trò M5 theo schema.py**: `Input: SELECT * FROM fact_gold_summary` — `Output: không ghi gì cả, chỉ hiển thị`. Notebook này KHÔNG để M5 tự sinh/ghi dữ liệu; phần "seed dữ liệu demo" ở dưới chỉ chạy tạm 3 module thật (M1/M2/M4) để có dữ liệu cho dashboard xem thử — trong dự án thật, 3 module đó chạy độc lập trên máy/tiến trình của M1/M2/M4.

## Chuẩn bị trước khi chạy (VS Code)
- Mở thư mục dự án trong VS Code, đặt notebook này **cùng cấp thư mục** với 6 file: `schema.py`, `m1_streaming.py`, `stochastic_simulator.py`, `kalman_filter.py`, `m2_analytics_pipeline.py`, `m4_ml_engineer.py`.
- Cài extension **Jupyter** (và **Python**) của Microsoft nếu chưa có.
- Chọn đúng kernel Python (venv/conda) ở góc trên phải notebook trước khi chạy cell đầu tiên.


## 1. Cài đặt thư viện

Chạy trong terminal tích hợp của VS Code (khuyên dùng, để thấy log cài đặt
đầy đủ) hoặc chạy thẳng cell dưới (dùng `%pip` để cài đúng vào kernel đang
chọn trong notebook, tránh cài nhầm vào Python hệ thống).

```bash
python -m venv .venv
# Windows: .venv\Scripts\activate
# macOS/Linux: source .venv/bin/activate
pip install streamlit pydantic scikit-learn pandas numpy
```


In [1]:
%pip install streamlit pydantic scikit-learn pandas numpy -q
print("✅ Đã cài đặt thư viện vào kernel hiện tại.")


Note: you may need to restart the kernel to use updated packages.
✅ Đã cài đặt thư viện vào kernel hiện tại.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Kiểm tra đủ 6 file thật của nhóm

Notebook này KHÔNG upload file qua trình duyệt (đó là tính năng riêng của
Colab). Trong VS Code, chỉ cần **copy 6 file thật** vào cùng thư mục với
notebook này:

`schema.py`, `m1_streaming.py`, `stochastic_simulator.py`, `kalman_filter.py`,
`m2_analytics_pipeline.py`, `m4_ml_engineer.py`

Cell dưới chỉ kiểm tra xem đã có đủ file chưa, báo rõ file nào còn thiếu.


In [2]:
import os

REQUIRED_FILES = [
    "schema.py",
    "m1_streaming.py",
    "stochastic_simulator.py",
    "kalman_filter.py",
    "m2_analytics_pipeline.py",
    "m4_ml_engineer.py",
]

cwd = os.getcwd()
missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]

if missing:
    raise FileNotFoundError(
        f"Thiếu {len(missing)} file trong thư mục hiện tại ({cwd}): {missing}\n"
        f"Hãy copy các file này vào cùng thư mục với notebook rồi chạy lại cell."
    )

print(f"✅ Đã có đủ 6 file trong: {cwd}")
for f in REQUIRED_FILES:
    print("  -", f)


✅ Đã có đủ 6 file trong: c:\Users\thien\code\mas\project\fix an vox
  - schema.py
  - m1_streaming.py
  - stochastic_simulator.py
  - kalman_filter.py
  - m2_analytics_pipeline.py
  - m4_ml_engineer.py


## 3. Tạo `app.py` — Dashboard M5 (đã sửa 4 điểm)

In [3]:
%%writefile app.py
import sqlite3
import time

import pandas as pd
import streamlit as st

# ==============================================================================
# 0. CẤU HÌNH — DB_PATH khớp đúng tên file mà M1/M2/M4 dùng (drone_fleet.db,
#    KHÔNG phải "DroneFleetDB.db" của bản cũ)
# ==============================================================================
DB_PATH = "drone_fleet.db"

# ==============================================================================
# 1. CẤU HÌNH NGÔN NGỮ (DICTIONARY TRANSLATION)
# ==============================================================================
CURRENT_LANG = "EN"  # Đổi thành "VI" nếu muốn giao diện tiếng Việt

LANG_MAP = {
    "VI": {
        "title": "🛸 Drone Fleet Monitoring Dashboard — M5 Workspace",
        "sidebar_header": "Cấu hình hệ thống",
        "select_drone": "Chọn Drone ID",
        "battery_label": "Năng lượng Pin",
        "motor_temp_label": "Nhiệt độ động cơ",
        "etr_label": "Dự báo thời gian bay còn lại (ETR)",
        "etr_help": "Dải tin cậy được tính toán bởi mô hình Quantile Regression của M4",
        "map_title": "📍 Bản đồ định vị Drone thực tế",
        "status_stable": "Ổn định",
        "status_normal": "Bình thường",
        "status_warning": "Nhiệt độ cao",
        "status_danger": "Cảnh báo yếu!",
        "minutes": "phút",
        "no_drones_title": "⚠️ Chưa có drone nào trong hệ thống",
        "no_drones_body": (
            "Bảng `dim_drones` đang trống hoặc database `{db}` chưa tồn tại. "
            "Hãy chạy M1 (streaming) trước để khởi tạo database và đăng ký drone."
        ),
        "no_gold_title": "⏳ Chưa có dữ liệu Gold cho drone này",
        "no_gold_body": (
            "M2/M4 chưa kịp tính xong cửa sổ Gold đầu tiên cho **{drone}**. "
            "Trang sẽ tự làm mới sau vài giây — không hiển thị dữ liệu giả."
        ),
        "last_window": "Cửa sổ dữ liệu gần nhất",
    },
    "EN": {
        "title": "🛸 Drone Fleet Monitoring Dashboard — M5 Workspace",
        "sidebar_header": "System Configuration",
        "select_drone": "Select Drone ID",
        "battery_label": "Battery Level",
        "motor_temp_label": "Motor Temperature",
        "etr_label": "Estimated Time Remaining (ETR)",
        "etr_help": "Confidence interval calculated by Member 4's Quantile Regression model",
        "map_title": "📍 Real-time Drone Telemetry Map",
        "status_stable": "Stable",
        "status_normal": "Normal",
        "status_warning": "High Temp",
        "status_danger": "Low Battery Alert!",
        "minutes": "mins",
        "no_drones_title": "⚠️ No drones registered yet",
        "no_drones_body": (
            "`dim_drones` is empty or `{db}` does not exist yet. "
            "Run M1 (streaming) first to initialize the database and register drones."
        ),
        "no_gold_title": "⏳ No Gold data yet for this drone",
        "no_gold_body": (
            "M2/M4 haven't finished computing the first Gold window for **{drone}** yet. "
            "This page auto-refreshes in a few seconds — no placeholder data is shown."
        ),
        "last_window": "Latest data window",
    },
}

text = LANG_MAP[CURRENT_LANG]

# ==============================================================================
# 2. KHỞI TẠO CONFIG TRANG STREAMLIT
# ==============================================================================
st.set_page_config(page_title="Drone Fleet Monitoring Dashboard", layout="wide")
st.title(text["title"])
st.sidebar.header(text["sidebar_header"])

# ==============================================================================
# 3. DANH SÁCH DRONE — LẤY ĐỘNG TỪ dim_drones, KHÔNG HARDCODE
#    (bản cũ hardcode ["DRONE_001", "DRONE_002"], bỏ sót DRONE_003)
# ==============================================================================
def get_drone_ids() -> list:
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("PRAGMA busy_timeout = 5000;")  # SỬA: tránh "database is locked"
        df = pd.read_sql_query("SELECT drone_id FROM dim_drones ORDER BY drone_id", conn)
        conn.close()
        return df["drone_id"].tolist()
    except Exception:
        # DB chưa tồn tại / bảng chưa tồn tại -> coi như chưa có drone nào,
        # KHÔNG fallback sang dữ liệu giả.
        return []


# SỬA: thêm hàm lấy profile (case study) của drone -- để dashboard trả lời
# được câu hỏi "VÌ SAO drone này đang cảnh báo", không chỉ hiển thị con số.
def get_drone_profile(drone_id: str) -> dict | None:
    try:
        conn = sqlite3.connect(DB_PATH)
        conn.execute("PRAGMA busy_timeout = 5000;")
        df = pd.read_sql_query(
            "SELECT * FROM dim_drones WHERE drone_id = ?", conn, params=(drone_id,)
        )
        conn.close()
    except Exception:
        return None
    if df.empty:
        return None
    return df.iloc[0].to_dict()


drone_ids = get_drone_ids()

if not drone_ids:
    st.warning(f"### {text['no_drones_title']}")
    st.write(text["no_drones_body"].format(db=DB_PATH))
    time.sleep(3)
    st.rerun()

drone_selected = st.sidebar.selectbox(text["select_drone"], drone_ids)

# ==============================================================================
# 3b. HIỂN THỊ PROFILE (CASE STUDY) — trả lời câu hỏi "VÌ SAO"
#     Trước đây payload/wind_zone/... chỉ tồn tại trong RAM lúc M3 chạy,
#     không ai xem lại được. Giờ M1 đã ghi profile vào dim_drones lúc đăng
#     ký fleet, nên M5 chỉ cần SELECT ra và hiển thị -- không tính toán gì
#     thêm (đúng vai trò M5: chỉ đọc).
# ==============================================================================
profile = get_drone_profile(drone_selected)
if profile:
    zone_note = []
    if profile.get("payload_kg", 0) > 0:
        zone_note.append(f"🏋️ Đang mang {profile['payload_kg']:.1f}kg hàng")
    if profile.get("wind_zone") == "storm":
        zone_note.append("🌪️ Đang bay trong vùng gió bão")
    elif profile.get("wind_zone") == "calm":
        zone_note.append("🍃 Vùng gió yên tĩnh")
    if profile.get("battery_health", 1.0) < 0.9:
        zone_note.append(f"🔋 Pin đã xuống cấp (health={profile['battery_health']:.2f})")
    if profile.get("ambient_temp_c", 25) < 15:
        zone_note.append(f"❄️ Thời tiết lạnh ({profile['ambient_temp_c']:.0f}°C)")
    if profile.get("gps_quality", 1.0) < 0.8:
        zone_note.append("📡 Module GPS chất lượng thấp")
    if profile.get("network_zone") == "rural":
        zone_note.append("📶 Vùng phủ sóng yếu (rural)")

    profile_label = profile.get("profile_label", "baseline")
    if zone_note:
        st.sidebar.info(f"**Kịch bản: `{profile_label}`**\n\n" + "\n".join(f"- {n}" for n in zone_note))
    else:
        st.sidebar.success(f"**Kịch bản: `{profile_label}`** (điều kiện tiêu chuẩn)")

# ==============================================================================
# 4. HÀM ĐỌC DỮ LIỆU TỪ TẦNG GOLD (M2 + M4 ghi, M5 CHỈ ĐỌC)
#    Giữ nguyên cách đọc ĐÚNG của bản cũ: lấy đúng 1 dòng mới nhất theo
#    window_end. KHÔNG còn fallback generate_mock_telemetry_row() — nếu
#    chưa có dữ liệu thì trả về None và UI tự xử lý, không giả vờ có số liệu.
# ==============================================================================
def get_gold_data(drone_id: str):
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA busy_timeout = 5000;")  # SỬA: tránh "database is locked"
    try:
        query = """
            SELECT * FROM fact_gold_summary
            WHERE drone_id = ?
            ORDER BY window_end DESC LIMIT 1
        """
        df = pd.read_sql_query(query, conn, params=(drone_id,))
    finally:
        conn.close()

    if df.empty:
        return None
    return df.iloc[0].to_dict()


data = get_gold_data(drone_selected)

if data is None:
    st.info(f"### {text['no_gold_title']}")
    st.write(text["no_gold_body"].format(drone=drone_selected))
    time.sleep(3)
    st.rerun()

st.caption(f"{text['last_window']}: {data.get('window_end', '—')}")

# ==============================================================================
# 5. HIỂN THỊ METRICS
# ==============================================================================
col1, col2, col3 = st.columns(3)

# SỬA: bản cũ chỉ check "== green" rồi coi mọi giá trị khác (kể cả "yellow")
# là đỏ nguy hiểm -> mất hết ý nghĩa của control limit 3 mức mà M2 tính bằng
# classify_status() (schema.py). Giờ map đủ 3 mức green/yellow/red.
STATUS_ICON_LABEL = {
    "green":  ("🟢", text["status_stable"]),
    "yellow": ("🟡", text["status_warning"]),
    "red":    ("🔴", text["status_danger"]),
}

with col1:
    icon, label = STATUS_ICON_LABEL.get(
        data.get("battery_status"), ("⚪", "—")
    )
    st.metric(
        label=f"{text['battery_label']} ({icon} {label})",
        value=f"{data.get('battery_mean', 0.0):.1f} %",
    )

with col2:
    icon, label = STATUS_ICON_LABEL.get(
        data.get("motor_temp_status"), ("⚪", "—")
    )
    st.metric(
        label=f"{text['motor_temp_label']} ({icon} {label})",
        value=f"{data.get('motor_temp_mean', 0.0):.1f} °C",
    )

with col3:
    etr_lower = data.get("etr_lower_min")
    etr_upper = data.get("etr_upper_min")
    if etr_lower is not None and etr_upper is not None:
        st.metric(
            label=text["etr_label"],
            value=f"{round(float(etr_lower), 1)} - {round(float(etr_upper), 1)} {text['minutes']}",
            help=text["etr_help"],
        )
    else:
        # M4 chưa kịp UPDATE 4 cột của mình cho cửa sổ này -> nói rõ là
        # đang chờ, không tự bịa công thức thay cho model của M4.
        st.metric(label=text["etr_label"], value="—", help=text["etr_help"])

st.markdown("---")

# ==============================================================================
# 6. BẢN ĐỒ — chỉ vẽ khi M4 đã có gps_lat_smooth/gps_lon_smooth, không tự
#    chế toạ độ mặc định như bản cũ (10.7626, 106.6602) khi thiếu dữ liệu.
# ==============================================================================
st.subheader(text["map_title"])

lat_smooth = data.get("gps_lat_smooth")
lon_smooth = data.get("gps_lon_smooth")

if lat_smooth is not None and lon_smooth is not None:
    map_data = pd.DataFrame([{"lat": lat_smooth, "lon": lon_smooth}])
    st.map(map_data)
else:
    st.info(text["no_gold_body"].format(drone=drone_selected))

# ==============================================================================
# 7. POLLING — near real-time, giữ nguyên cơ chế đúng của bản cũ
# ==============================================================================
time.sleep(3)
st.rerun()


Writing app.py


## 4. (Demo) Seed dữ liệu bằng chính pipeline thật M1 → M2 → M4

⚠️ Đây **không phải việc của M5** trong kiến trúc thật (M5 chỉ đọc
`fact_gold_summary`). Cell dưới chỉ chạy tạm 3 module thật của nhóm ngay
trong notebook để dashboard có dữ liệu mà xem thử. Trong dự án thật, M1
chạy streaming liên tục, M2/M4 chạy pipeline riêng (`--loop`) trên tiến
trình/máy khác — M5 không cần quan tâm chúng chạy ở đâu, chỉ cần chung 1
file `drone_fleet.db`.


In [5]:
import importlib
import queue as pyqueue
import sqlite3
import threading
import time

import m1_streaming as m1
import m2_analytics_pipeline as m2
import m4_ml_engineer as m4

importlib.reload(m1)
importlib.reload(m2)
importlib.reload(m4)

# ---- Bước 1: M1 — sinh dữ liệu Bronze bằng chính producer/consumer thật ----
# SỬA: dùng đúng kiến trúc asyncio thật của m1_streaming.py (producer_loop_thread
# chạy 1 thread DUY NHẤT chứa nhiều coroutine), không phải "1 thread/drone" của
# kiến trúc cũ (hàm m1.producer_loop không tồn tại trong bản mới -> bản trước
# sẽ crash AttributeError ngay tại đây).
#
# SỬA: fleet demo trong notebook dùng SỐ NHỎ (5 drone), không dùng
# m1.FLEET_CONFIG mặc định (5.000 drone) — quy mô đó chỉ hợp lý khi chạy
# m1_streaming.py độc lập ngoài terminal cho demo tải lớn, không phải cho
# 1 cell notebook seed dữ liệu để xem thử dashboard.
DEMO_NUM_DRONES = 5
demo_fleet = m1.generate_fleet_config(num_drones=DEMO_NUM_DRONES, weak_signal_every=0)

m1.init_db(m1.DB_PATH, demo_fleet)

sim_maker = m1.get_sim_output_maker()  # USE_MOCK_DATA=True mặc định trong m1_streaming.py
data_queue: "pyqueue.Queue" = pyqueue.Queue(maxsize=1000)
stop_event = threading.Event()
stats = m1.ConsumerStats()

consumer_thread = threading.Thread(
    target=m1.consumer_loop,
    args=(data_queue, m1.DB_PATH, stop_event, stats),
    name="Consumer-Writer",
)
# 1 THREAD DUY NHẤT chạy toàn bộ event loop asyncio cho cả demo_fleet — đây là
# đúng hàm mà m1.main() thật sự dùng, không phải "1 thread/drone".
producer_thread = threading.Thread(
    target=m1.producer_loop_thread,
    args=(demo_fleet, sim_maker, data_queue, stop_event),
    name="Producer-AsyncIO",
)

consumer_thread.start()
producer_thread.start()

SEED_SECONDS = 15
print(f"[M1] Đang stream dữ liệu demo trong {SEED_SECONDS}s cho {len(demo_fleet)} drone...")
time.sleep(SEED_SECONDS)
stop_event.set()
producer_thread.join(timeout=5)
consumer_thread.join(timeout=5)
print(f"[M1] Xong. Đã ghi {stats.written} dòng Bronze, {stats.rejected} dòng bị từ chối.")

# ---- Bước 2: M2 — Bronze -> Silver -> Gold ----
conn = sqlite3.connect(m1.DB_PATH)
conn.execute("PRAGMA journal_mode=WAL;")
m2.init_db(conn)
m2.run_once(conn)
conn.close()
print("[M2] Đã chạy Bronze -> Silver -> Gold xong.")

# ---- Bước 3: M4 — Kalman smoothing + ETR quantile regression ----
conn = sqlite3.connect(m1.DB_PATH)
try:
    df = m4.load_fact_telemetry(conn)
    print(f"[M4] Đọc được {len(df)} dòng fact_telemetry.")
    model_lower_baseline, model_upper_baseline = m4.train_linear_quantile_baseline(df)
    df_smoothed = m4.apply_kalman_smoothing(df)
    windows = m4.build_window_aggregates(df_smoothed, window_minutes=5)
    existing_gold_windows = m4.load_existing_gold_windows(conn)
    m4.predict_and_update(
    conn, windows,
    model_lower, model_upper,
    model_lower_baseline, model_upper_baseline,
    existing_gold_windows,
    dry_run=False,
)
except ValueError as e:
    print(f"[M4] Chưa đủ dữ liệu để train ({e}).")
    print("     -> Tăng SEED_SECONDS ở trên rồi chạy lại cell này.")
finally:
    conn.close()

print("✅ Đã seed xong dữ liệu demo — có thể mở dashboard ở bước dưới.")


2026-07-18 21:48:53,676 [INFO] MainThread: Database sẵn sàng tại 'drone_fleet.db', đã đăng ký 5 drone (kèm profile)
2026-07-18 21:48:53,685 [INFO] Consumer-Writer: Consumer (writer) bắt đầu, đã kết nối database
2026-07-18 21:48:53,697 [INFO] Producer-AsyncIO: DRONE_00001 -> profile: baseline (payload=0.0kg, wind_zone=moderate, battery_health=1.0, ambient=25.0C)
2026-07-18 21:48:53,701 [INFO] Producer-AsyncIO: DRONE_00002 -> profile: heavy_payload (payload=4.0kg, wind_zone=moderate, battery_health=1.0, ambient=25.0C)
2026-07-18 21:48:53,702 [INFO] Producer-AsyncIO: DRONE_00003 -> profile: storm_zone (payload=0.0kg, wind_zone=storm, battery_health=1.0, ambient=25.0C)
2026-07-18 21:48:53,703 [INFO] Producer-AsyncIO: DRONE_00004 -> profile: heavy_in_storm (payload=4.0kg, wind_zone=storm, battery_health=1.0, ambient=25.0C)
2026-07-18 21:48:53,705 [INFO] Producer-AsyncIO: DRONE_00005 -> profile: aging_battery (payload=0.0kg, wind_zone=moderate, battery_health=0.75, ambient=25.0C)
2026-07-18 

[INFO] Đã import KalmanFilter2D thật từ kalman_filter.py (M3)
[M1] Đang stream dữ liệu demo trong 15s cho 5 drone...


2026-07-18 21:49:08,844 [INFO] Consumer-Writer: Consumer đã dừng. Tổng kết: 75 dòng ghi thành công, 0 dòng bị từ chối
2026-07-18 21:49:08,855 [INFO] MainThread: Đang xử lý 75 bản ghi Bronze mới (log_id > 973583)...
2026-07-18 21:49:08,877 [INFO] MainThread: Bronze -> Silver: 75 dòng ghi thành công, 0 dòng bị từ chối.
2026-07-18 21:49:08,879 [INFO] MainThread: Đang cập nhật 5 cửa sổ Gold bị ảnh hưởng...


[M1] Xong. Đã ghi 75 dòng Bronze, 0 dòng bị từ chối.


2026-07-18 21:49:09,971 [INFO] MainThread: Gold: đã UPSERT 5 cửa sổ (drone_id, window_end).


[M2] Đã chạy Bronze -> Silver -> Gold xong.
[M4] Đọc được 973658 dòng fact_telemetry.
[INFO] Train xong Linear Quantile Regression (baseline) trên 973658 dòng.
[INFO] Đã UPDATE 5010 dòng trong fact_gold_summary (mô hình chính + baseline).
✅ Đã seed xong dữ liệu demo — có thể mở dashboard ở bước dưới.


## 5. Chạy Dashboard (local, không cần tunnel)

Vì chạy trên máy bạn (VS Code), Streamlit không cần Bore tunnel như trên
Colab — chỉ cần mở `http://localhost:8501` là xong.

**Cách khuyên dùng**: mở terminal tích hợp của VS Code (Terminal ▸ New
Terminal), `cd` vào đúng thư mục notebook, rồi chạy:

```bash
streamlit run app.py
```

VS Code sẽ tự nhận cổng `8501` và cho bạn bấm mở trong trình duyệt; log của
Streamlit cũng hiện trực tiếp trong terminal đó (dễ debug hơn chạy ngầm
trong notebook).

Nếu muốn chạy ngay trong notebook (không mở terminal riêng), dùng cell dưới
— nó chạy Streamlit ở tiến trình nền và tự mở trình duyệt.


In [8]:
import subprocess
import sys
import time
import webbrowser

PORT = 8501

streamlit_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", str(PORT), "--server.headless", "true"],
)

time.sleep(4)  # đợi Streamlit khởi động
url = f"http://localhost:{PORT}"
print(f"✅ Dashboard đang chạy tại: {url}")
webbrowser.open(url)
print("Tiến trình chạy ngầm (PID:", streamlit_proc.pid, "). Chạy cell bên dưới để dừng khi xong.")


✅ Dashboard đang chạy tại: http://localhost:8501
Tiến trình chạy ngầm (PID: 30352 ). Chạy cell bên dưới để dừng khi xong.


In [9]:
# Chạy cell này khi muốn TẮT dashboard đang chạy ngầm ở trên
try:
    streamlit_proc.terminate()
    print("Đã dừng Streamlit.")
except NameError:
    print("Không tìm thấy tiến trình Streamlit nào đang chạy từ notebook này.")


Đã dừng Streamlit.
